# Journal de crawl — corpus « airegulation » (Art. 1, *Does Extraction Level Matter?*)

> **Nature du document.** Journal de production du corpus : commandes réellement exécutées,
> vagues de crawl, décisions de périmètre, incidents et correctifs, plus les audits SQL qui
> permettent de re-vérifier chaque chiffre en base. Dérivé du tutoriel A→Z de MWI, il n'en garde
> que ce qui a été fait — les étapes **non exécutées sont explicitement marquées**. Il complète
> l'audit-trail interne du projet (journal de developpement non publie).
>
> **Fenêtre couverte** : 11/06/2026 (création du land) → 04/07/2026 (dédup des URL + exports appariés).
> **Instrument** : MyWebIntelligencePython, clone `02_data/mwi`, HEAD `cd850ea` (04/07/2026).

### Le design servi par ce corpus

L'étude compare, **sur un même corpus crawlé**, deux procédures d'extraction de liens : page
entière (réseau A, export `--fullhtml`) contre corps de texte éditorial (réseau B, readable).
Terrain : la controverse sur la régulation de l'IA dans le web anglophone (2023-2026). Le land a
été créé avec `--fullhtml=TRUE` : le HTML brut de chaque page est archivé, les deux extractions
sont rejouables *offline* sur les mêmes documents.

### État réel du corpus (audit SQL du 04/07/2026)

| Dimension | Valeur constatée |
|---|---|
| Land | `airegulation` (id 1, `lang=en`, fullhtml=ON), créé le 11/06/2026 10:16 |
| Dictionnaire | 57 termes (4 axes de la controverse) |
| Expressions | 165 564 : 21 712 crawlées (profondeur 0) + 143 852 non crawlées (profondeur 1 = frontière) |
| Crawl | 7 journées entre le 11/06 et le 30/06 ; HTTP 200 : 20 806 ; aiohttp 16 193 / curl_cffi 3 908 / archive_org 1 611 |
| Texte éditorial | 20 114 pages avec readable ; 20 699 avec HTML brut archivé |
| Qualification | 21 430 pages à pertinence ≥ 1 ; gate LLM : 11 682 « oui » ; 282 pages à pertinence 0 conservées |
| Enrichissements | SEO Rank : 14 188 pages ; domaines fetchés : 25 010 (table globale : 38 989) |
| Liens body-text | 194 013 lignes `expressionlink` (`context` : 193 449 ; `dom` : 190 810) |
| Graphe apparié (exports du 04/07, minrel=1) | 21 430 nœuds ; A = 22 778 arêtes ; B = 8 152 ; `citation=1` = 8 205 |
| **Non exécuté** | crawl profondeur ≥ 1, analyse des médias, embeddings/pseudolinks, tags |

### Écarts au design initial (à acter dans §4 Methods du manuscrit)

1. **Profondeur** : le design prévoyait un crawl jusqu'à profondeur 6 ; le corpus est arrêté à la
   **profondeur 0** (les graines). Les 143 852 URL de profondeur 1 découvertes dans les pages
   restent non crawlées : elles sont les **cibles** du réseau, jamais des sources.
2. **Requêtes** : 8 requêtes en 4 axes étaient prévues ; le journal n'atteste que **Q1 et Q2
   exécutées** (×2 moteurs chacune), Q3/Q4 définies (exécution à confirmer — §4).
3. **Volume** : 8 000–15 000 pages visées → 21 712 crawlées, 21 430 qualifiées.

### Table des matières

- [§0 — Conventions et amorçage](#sec0)
- [§1 — Instrument](#sec1)
- [§2 — Le Land](#sec2)
- [§3 — Le dictionnaire](#sec3)
- [§4 — Collecter les graines](#sec4)
- [§5 — Le crawl](#sec5)
- [§6 — Normaliser les URLs](#sec6)
- [§7 — Texte éditorial (readable)](#sec7)
- [§8 — Qualification et nettoyage](#sec8)
- [§9 — Domaines](#sec9)
- [§10 — SEO Rank](#sec10)
- [§11 — Médias (non exécuté)](#sec11)
- [§12 — Consolidation et dédup des URL (03–04/07)](#sec12)
- [§13 — Embeddings (non exécuté)](#sec13)
- [§14 — Tags (à venir)](#sec14)
- [§15 — Exports](#sec15)
- [§16 — Bilan chiffré et reproductibilité](#sec16)
- [§17 — Récapitulatif et reste-à-faire](#sec17)

<a id="sec0"></a>
## §0 — Conventions et amorçage

Trois types de cellules :

1. **Cellules `!commande`** — le CLI MyWebIntelligence (`python mywi.py …`), telles qu'exécutées.
   Les opérations lourdes ont parfois été lancées en terminal plutôt qu'ici ; le journal reproduit
   alors la commande dans un bloc de code, avec sa date.
2. **Cellules SQL d'audit** — lecture seule via le helper `sql()` ; chaque chiffre de ce journal
   se re-vérifie par elles.
3. **Notes datées** — décisions, incidents, correctifs. Les 💡/⚠️ du tutoriel d'origine ne sont
   conservés que lorsqu'ils documentent un piège réellement rencontré sur ce corpus.

> ⚠️ **Prérequis** : `jupyter` et `pandas` dans l'environnement de MWI (`pip install jupyter pandas`).
> Le notebook vit dans `02_data/`, le dépôt dans `02_data/mwi/` — la cellule d'amorçage gère les deux.

In [ ]:
import os, sys, json, glob, sqlite3
from pathlib import Path
from datetime import datetime

# Localiser la racine du dépôt — le notebook vit dans 02_data/, le dépôt dans 02_data/mwi/
_here = Path.cwd()
for cand in (_here, _here / "mwi", _here.parent, _here.parent / "mwi"):
    if (cand / "mywi.py").exists():
        ROOT = cand
        break
else:
    raise SystemExit("Impossible de trouver mywi.py — lancer ce notebook depuis 02_data/ ou depuis le dépôt mwi/")
os.chdir(ROOT)

# Emplacement des données depuis settings.py (override possible via MYWI_DATA_DIR)
sys.path.insert(0, str(ROOT))
import settings
DATA = Path(os.environ.get("MYWI_DATA_DIR", settings.data_location)).expanduser()
if not DATA.is_absolute():
    DATA = ROOT / DATA
DB = DATA / "mwi.db"

LAND = "airegulation"

try:
    import pandas as pd
except ImportError:
    raise SystemExit("pandas est requis pour les audits SQL — pip install pandas")
pd.set_option("display.max_colwidth", 120)

def sql(query, params=()):
    """Audit SQL en lecture seule. Le fallback immutable contourne le verrouillage
    capricieux de Google Drive — ne l'utiliser qu'à base au repos (aucun crawl en cours)."""
    last = None
    for uri in (f"file:{DB}?mode=ro", f"file:{DB}?mode=ro&immutable=1"):
        try:
            with sqlite3.connect(uri, uri=True) as conn:
                return pd.read_sql_query(query, conn, params=params)
        except sqlite3.OperationalError as e:
            last = e
    raise last

print(f"Racine du dépôt : {ROOT}")
print(f"Base SQLite     : {DB}  ({'existe' if DB.exists() else 'ABSENTE'})")
print(f"Land du projet  : {LAND}")

<a id="sec1"></a>
## §1 — Instrument

| Contrôle | Commande | État constaté (04/07/2026) |
|---|---|---|
| Schéma de base | `db migrate` | ⚠️ Base migrée jusqu'à **012** ; la migration **013** (index composites `expression`, commit `c60246c` du 24/06) **n'a jamais été appliquée** — relancer `db migrate` (idempotent) |
| Fournisseurs de recherche | `search check` | Collecte passée par la voie legacy `land urlist` (SerpAPI + DuckDuckGo) ; le routeur v2 `search run` n'a pas été utilisé |
| Mercury Parser | — | Non sollicité : `land readable` a extrait localement (Trafilatura) depuis le HTML archivé |

> ⚠️ **Jamais `db setup` sur une base existante** : destructif (drop + recreate de toutes les
> tables). La seule commande légitime sur cette base est `db migrate`.

In [ ]:
# Mise à niveau du schéma — IDEMPOTENT. À relancer : la migration 013 (index) est en attente sur cette base.
!python mywi.py db migrate

In [ ]:
# Fournisseurs configurés (clé absente = fournisseur ignoré silencieusement)
!python mywi.py search check

In [ ]:
# Audit : migrations réellement appliquées (constat du 04/07 : 001 → 012, la 013 manque)
sql("SELECT * FROM schema_migrations ORDER BY 1")

<a id="sec2"></a>
## §2 — Le Land

Créé le **11/06/2026 à 10:16**. Les trois décisions de création découlent du design :

| Paramètre | Valeur | Justification |
|---|---|---|
| `--name` | `airegulation` | Identifiant stable, réutilisé dans toutes les commandes |
| `--lang` | `en` | Terrain anglophone : pilote le stemmer du score de pertinence ; page détectée dans une autre langue → pertinence forcée à 0 |
| `--fullhtml` | `TRUE` | **Non négociable** : le design apparié exige le HTML brut de chaque page pour rejouer les deux extractions sur les mêmes documents (~100 Ko/page, plafond `settings.fullhtml_max_size_kb`) |

> 💡 La description ci-dessous est celle **réellement en base** : rédigée en anglais et calibrée
> pour servir de consigne à la gate LLM de qualification (§8) — elle définit l'inclusion (lois,
> politiques, risques, éthique, lobbying, prises de position sur la gouvernance de l'IA) et
> l'exclusion (tutoriels techniques, annonces produit, actualité IA sans angle régulatoire).

In [ ]:
DESC = ("Public debate and controversy over the regulation and governance of artificial intelligence (2023-2026): "
        "institutional regulation (EU AI Act, US executive orders, national AI policies), societal risks of AI "
        "(bias, discrimination, algorithmic harm, surveillance), AI safety and existential risk debates, corporate "
        "accountability and transparency (open vs closed models, industry self-governance), and copyright or "
        "intellectual property disputes over generative AI. Relevant pages discuss laws, policies, risks, ethics, "
        "lobbying, or public positions on AI governance — not purely technical AI tutorials, product announcements, "
        "or AI news without a regulatory or societal angle.")

!python mywi.py land create --name={LAND} --desc="{DESC}" --lang=en --fullhtml=TRUE

> ⚠️ Le land existant, une ré-exécution échoue sur la contrainte d'unicité — normal, ne pas
> re-créer. Vérification de la fiche :

In [ ]:
!python mywi.py land list --name={LAND}

<a id="sec3"></a>
## §3 — Le dictionnaire du Land

Le vocabulaire de pertinence du projet :

```text
relevance = 10 × (occurrences des lemmes dans le titre) + 1 × (occurrences dans le texte)
```

Pertinence forcée à **0** hors langue `en` ou sur verdict négatif de la gate LLM (§8).
Lemmatisation Snowball anglaise : `regulation`/`regulations` se replient sur le même lemme
(`regul`) ; les variantes dérivationnelles (`regulatory` → `regulatori`) sont listées
explicitement.

**État réel : 57 termes**, couvrant les 4 axes de la controverse (régulation institutionnelle ;
risques sociétaux et sécurité ; gouvernance industrielle ; propriété intellectuelle). Ajoutés en
plusieurs passes autour de la création du land ; la commande ci-dessous reconstitue le
vocabulaire complet tel qu'il est en base.

> 💡 `land addterm` recalcule immédiatement la pertinence de toutes les expressions du Land —
> instantané sur un land vide, plusieurs minutes après le crawl de 21 712 pages.

In [ ]:
!python mywi.py land addterm --land={LAND} --terms="artificial intelligence, AI act, AI regulation, AI governance, AI policy, AI law, AI legislation, AI oversight, regulatory framework, executive order, federal preemption, state AI laws, deregulation, overregulation, regulatory burden, AI compliance, AI enforcement, AI safety, existential risk, frontier model, frontier AI, AI alignment, superintelligence, catastrophic risk, AI risks, algorithmic bias, algorithmic discrimination, AI harms, civil rights, algorithmic accountability, AI transparency, facial recognition, deepfake, AI surveillance, self-regulation, voluntary commitments, code of practice, general-purpose AI, GPAI, systemic risk, frontier model forum, responsible AI, open source AI, open weights, safety framework, red teaming, AI copyright, fair use, training data, intellectual property, content licensing, licensing deal, copyright infringement, copyright lawsuit, rightsholders, web scraping, generative AI"

In [ ]:
# Audit : les 57 termes et leurs lemmes tels qu'en base
sql("""SELECT w.* FROM landdictionary ld JOIN word w ON w.id = ld.word_id
       WHERE ld.land_id = 1 ORDER BY w.id""")

<a id="sec4"></a>
## §4 — Collecter les graines

La collecte est passée par la **voie legacy `land urlist`** : balayage par **fenêtres mensuelles
datées** (juin 2023 → mai 2026), sur **Google (SerpAPI)** puis **DuckDuckGo**, à requête
constante. La séquence législative 2023-2026 étant au cœur du terrain, le balayage temporel
protège la couverture des épisodes anciens que le classement courant des moteurs aurait enterrés.

> ⚠️ **Limite d'instrument à documenter dans l'article** : `land urlist` **ne journalise pas en
> base** — les tables `searchquery`/`searchresultlog` (réservées au routeur v2 `search run`,
> jamais utilisé ici) sont **vides**, et `expression` ne porte pas de colonne de provenance.
> **Ce notebook est donc l'unique journal des requêtes de collecte.** Méthode de contrôle
> appliquée : relancer chaque requête jusqu'à zéro nouvelle entrée (fait ×3).

In [ ]:
Q1 = '"EU AI Act" regulation'
Q2 = '"US AI Act" regulation'
Q3 = '("artificial intelligence" OR "intelligence artificielle" OR AI OR IA) AND ("AI Act" OR AIA OR regulation OR legislation OR governance) AND ("European Union" OR EU OR "Union européenne" OR UE OR Europe)'
Q4 = '("artificial intelligence" OR "intelligence artificielle" OR AI OR IA) AND (regulation OR legislation OR governance OR policy OR "executive order") AND ("United States" OR "États-Unis" OR US OR USA OR federal)' 

### Exécution (×2 moteurs par requête)

Chaque requête est balayée de `2023-06-01` à `2026-05-31` par pas d'un mois, 1 s de pause entre
fenêtres. Sans `--gl` : scope langue seul (`hl=en&lr=lang_en`) — corpus anglophone **global**,
sans biais pays. La persistance est fenêtre par fenêtre : un échec SerpAPI transitoire ne perd
que la fenêtre en cours.

> 💡 Sous IPython, `'${Q1}'` et `'{Q2}'` sont équivalents : les deux formes substituent la
> variable Python — les requêtes envoyées étaient bien les chaînes définies ci-dessus.

In [ ]:
!python mywi.py land urlist --name={LAND} --query='${Q1}' --engine=google --lang=en --datestart=2023-06-01 --dateend=2026-05-31 --timestep=month --sleep=1.0

!python mywi.py land urlist --name={LAND} --query='${Q1}' --engine=duckduckgo --lang=en --datestart=2023-06-01 --dateend=2026-05-31 --timestep=month --sleep=1.0

!python mywi.py land urlist --name={LAND} --query='{Q2}' --engine=google --lang=en --datestart=2023-06-01 --dateend=2026-05-31 --timestep=month --sleep=1.0

!python mywi.py land urlist --name={LAND} --query='{Q2}' --engine=duckduckgo --lang=en --datestart=2023-06-01 --dateend=2026-05-31 --timestep=month --sleep=1.0

> ⚠️ **Trous du journal, à confirmer par l'opérateur** (aucune trace en base ne peut trancher) :
> 1. **Q3 et Q4** : définies ci-dessus mais aucune cellule d'exécution conservée. Le volume final
>    de graines (21 712 pages de profondeur 0) excède vraisemblablement ce que Q1+Q2 seules
>    produisent sur 36 fenêtres × 2 moteurs — Q3/Q4 ont probablement été lancées (en terminal ?).
>    À confirmer et, le cas échéant, consigner ici la commande exacte.
> 2. Moteurs additionnels (`bing` ?) ou compléments manuels (`land addurl`) éventuels.

In [ ]:
# Audit : les graines = expressions de profondeur 0 (toutes crawlées au 04/07)
sql("""SELECT depth, COUNT(*) AS n, SUM(fetched_at IS NOT NULL) AS fetchees
       FROM expression WHERE land_id = 1 GROUP BY depth""")

<a id="sec5"></a>
## §5 — Le crawl

Le crawl télécharge chaque page, extrait métadonnées, liens (créés en profondeur +1) et médias,
calcule la pertinence, et archive le HTML brut (politique fullhtml du land). Cascade
anti-blocage : `aiohttp` → `curl_cffi` (empreinte TLS Chrome) → `playwright` (opt-in) → archive
Wayback. Deux colonnes se lisent **ensemble** : `http_status` = statut de la stratégie **qui a
livré** le HTML (un 403 sauvé par curl_cffi devient 200) ; `fetch_method` = la stratégie
livreuse. Le signal « le serveur d'origine m'a bloqué » = `fetch_method != 'aiohttp'`.

### 5.1 — Exécution réelle : rafales bornées, profondeur 0 uniquement

Le crawl a été lancé **en terminal** (pas dans ce notebook), en rafales de 100 × 100 pages :

```bash
for i in {1..100}; do python mywi.py land crawl --name="airegulation" --depth=0 --limit=100; done
```

relancées sur plusieurs jours, jusqu'à épuisement des graines :

| Jour | Pages fetchées |
|---|---|
| 11/06/2026 | 1 702 |
| 12/06/2026 | 9 534 |
| 13/06/2026 | 1 769 |
| 14/06/2026 | 1 112 |
| 27/06/2026 | 21 |
| 29/06/2026 | 6 727 |
| 30/06/2026 | 847 |
| **Total** | **21 712** |

> 📌 **Décision de périmètre (à acter dans §4 Methods)** : **profondeur 0 uniquement** — le
> design initial prévoyait 6. Les 143 852 expressions de profondeur 1 créées par le crawl
> (liens découverts) restent volontairement non crawlées : elles constituent la **frontière**
> du corpus — cibles légitimes du réseau de citations, jamais sources.

### 5.2 — Audit de la collecte

`land list` donne la photographie d'ensemble (statuts, méthodes, volume HTML archivé) ; les
cellules SQL re-dérivent les chiffres du journal depuis la base.

In [ ]:
!python mywi.py land list --name={LAND}

**Instantané historique (fin juin, avant nettoyage)** — sortie `land list` collée telle
quelle. On y lit l'état **avant** `land delete` et la dédup du 04/07 : 207 366 expressions
(dont 179 112 non crawlées), 26 322 pages en 200, 26 118 HTML archivés (7 325 Mo), 57 termes.
À comparer au bilan courant (§16).

```text
(.venv) macbook-pro-de-amar3:mwi amarlakel$ python mywi.py land list --name=airegulation
airegulation - (June 11 2026 10:16)
        Public debate and controversy over the regulation and governance of artificial intelligence (2023-2026): institutional regulation (EU AI Act, US executive orders, national AI policies), societal risks of AI (bias, discrimination, algorithmic harm, surveillance), AI safety and existential risk debates, corporate accountability and transparency (open vs closed models, industry self-governance), and copyright or intellectual property disputes over generative AI. Relevant pages discuss laws, policies, risks, ethics, lobbying, or public positions on AI governance — not purely technical AI tutorials, product announcements, or AI news without a regulatory or societal angle.
        57 terms in land dictionary ['artificial intelligence', 'AI act', 'AI regulation', 'AI governance', 'AI policy', 'AI law', 'AI legislation', 'AI oversight', 'regulatory framework', 'executive order', 'federal preemption', 'state AI laws', 'deregulation', 'overregulation', 'regulatory burden', 'AI compliance', 'AI enforcement', 'AI safety', 'existential risk', 'frontier model', 'frontier AI', 'AI alignment', 'superintelligence', 'catastrophic risk', 'AI risks', 'algorithmic bias', 'algorithmic discrimination', 'AI harms', 'civil rights', 'algorithmic accountability', 'AI transparency', 'facial recognition', 'deepfake', 'AI surveillance', 'self-regulation', 'voluntary commitments', 'code of practice', 'general-purpose AI', 'GPAI', 'systemic risk', 'frontier model forum', 'responsible AI', 'open source AI', 'open weights', 'safety framework', 'red teaming', 'AI copyright', 'fair use', 'training data', 'intellectual property', 'content licensing', 'licensing deal', 'copyright infringement', 'copyright lawsuit', 'rightsholders', 'web scraping', 'generative AI']
        207366 expressions in land (179112 remaining to crawl)
        Status codes: 000: 77 - 200: 26322 - 202: 62 - 307: 1 - 400: 1 - 401: 46 - 402: 13 - 403: 1167 - 404: 55 - 405: 8 - 406: 5 - 415: 1 - 418: 1 - 429: 21 - 451: 2 - 455: 6 - 466: 1 - 500: 5 - 502: 3 - 503: 10 - 504: 1 - 520: 1 - 999: 304 - ERR: 141
        Fetch methods: aiohttp: 21461 - curl_cffi: 4831 - archive_org: 1962
        Full HTML: policy=ON — 26118 expressions stored (7325.1 MB)
        Embedding: paragraph: 0 - embed: 0 - pseudolink: 0
```

In [ ]:
# Audit : vagues de crawl, statuts HTTP, méthodes de fetch (état courant de la base)
display(sql("""SELECT date(fetched_at) AS jour, COUNT(*) AS pages FROM expression
               WHERE land_id = 1 AND fetched_at IS NOT NULL GROUP BY jour ORDER BY jour"""))
display(sql("""SELECT http_status, COUNT(*) AS n FROM expression
               WHERE land_id = 1 AND fetched_at IS NOT NULL
               GROUP BY http_status ORDER BY n DESC LIMIT 12"""))
display(sql("""SELECT fetch_method, COUNT(*) AS n FROM expression
               WHERE land_id = 1 AND fetched_at IS NOT NULL
               GROUP BY fetch_method ORDER BY n DESC"""))

### 5.3 — Rattrapage des blocages : `--retry-status`

`--retry-status=403,429` rejoue la cascade complète sur ces statuts, en ignorant le filtre
« jamais crawlé ». Sur ce corpus, la cascade a livré 3 908 pages via curl_cffi et 1 611 via
l'archive Wayback ; **653 pages restent en 403** — du Cloudflare Enterprise pour l'essentiel,
qui résiste à curl_cffi comme à Playwright : **limite d'instrument à documenter dans l'article,
pas à contourner**.

> ⚠️ À confirmer : la commande ci-dessous a-t-elle été passée (la micro-vague du 27/06 —
> 21 pages — y ressemble) ? Consigner la date en cas de relance.

In [ ]:
!python mywi.py land crawl --name={LAND} --retry-status=403,429

<a id="sec6"></a>
## §6 — Normaliser les URLs

Un corpus réel accumule des doublons d'URL (`http://` vs `https://`, `www.` vs nu, trackers,
slash final…). Or le design compte des **entités** : deux URLs pour la même page faussent nœuds
et arêtes. `land normalize` applique rétrospectivement `settings.url_normalization` : **rename**
(forme canonique nouvelle → mise à jour en place, original archivé dans `original_url`) ou
**merge** (forme canonique déjà présente → liens remappés vers l'expression canonique, doublon
supprimé en cascade).

**Trace en base** : 121 842 expressions portent un `original_url` (17 551 en profondeur 0,
104 291 en profondeur 1). La passe décisive de **dédup des variantes** (slash/scheme/www/casse),
qui a fait converger les deux définitions du réseau B, est journalisée en **§12**.

> ⚠️ Règle d'or appliquée : jamais de `normalize` sans backup ni `--dry-run` préalable.

In [ ]:
# 1) Aperçu sans modification — que ferait la normalisation ?
!python mywi.py land normalize --name={LAND} --dry-run --verbose

In [ ]:
# 2) Application réelle (après validation de l'aperçu)
!python mywi.py land normalize --name={LAND}
# Variante : --reset-status remet http_status/fetched_at à NULL pour re-crawler les URLs renommées

### 6.1 — Domaines `web.archive.org`

Les pages sauvées par la stratégie Wayback peuvent se retrouver rattachées au domaine
`web.archive.org` au lieu de leur domaine d'origine — ce qui fausserait l'agrégation par entité.
`db fix_archive_domains` corrige ce rattachement (idempotent). **Constat du 04/07 : 25
expressions encore rattachées à un domaine archive.org** (sur 1 611 pages fetchées via Wayback) —
résidu marginal, à refixer ou à documenter.

In [ ]:
# Aperçu d'abord (--dryrun est un vrai flag booléen ici), puis application
!python mywi.py db fix_archive_domains --dryrun
!python mywi.py db fix_archive_domains

<a id="sec7"></a>
## §7 — Extraire le texte éditorial (`readable`)

**Le cœur méthodologique de l'étude** : la Procédure B repose sur la séparation entre corps de
texte éditorial et appareillage de page (navigation, pieds de page, encarts, publicité).
Pipeline : (1) HTML archivé présent (notre cas, fullhtml=ON) → extraction **locale Trafilatura**,
sans re-téléchargement — le même HTML donne toujours le même texte ; (2) sinon Mercury Parser
sur l'URL ; (3) fusion `--merge=smart_merge` ; (4) les liens du markdown deviennent des
`expressionlink`, avec `context` (paragraphe d'énonciation) et `dom` (chemin DOM) — migration
012 ; (5) recalcul de la pertinence.

**Constat** : 20 114 pages avec readable sur 21 712 crawlées (92,6 %) ; **194 013 liens
éditoriaux** en `expressionlink`, dont 193 449 avec `context` et 190 810 avec `dom`.

> 📌 Le choix d'extracteur (Trafilatura, Mercury en secours) est **à documenter dans §4
> Methods** : la sensibilité du « body-text » à l'extracteur de readable est une objection
> reviewer attendue.

In [ ]:
!python mywi.py land readable --name={LAND} --merge=smart_merge

In [ ]:
# Audit : couverture readable + liens éditoriaux
display(sql("""SELECT COUNT(*) AS crawlees,
                      SUM(readable IS NOT NULL AND readable <> '') AS avec_readable,
                      SUM(html IS NOT NULL) AS avec_html
               FROM expression WHERE land_id = 1 AND fetched_at IS NOT NULL"""))
display(sql("""SELECT COUNT(*) AS liens, SUM(context IS NOT NULL) AS avec_context,
                      SUM(dom IS NOT NULL) AS avec_dom
               FROM expressionlink"""))

<a id="sec8"></a>
## §8 — Qualifier et nettoyer le corpus

Trois instruments, du moins cher au plus cher : le **score lexical** (gratuit, transparent,
grossier), la **gate LLM** (`land llm validate` — verdict oui/non par page, sur la base du
readable et de la description du land, d'où la description-consigne de §2), la **suppression
contrôlée** (`land delete --maxrel`, irréversible, en dernier).

**Exécution réelle** :
- Gate LLM passée **à l'échelle** (pas seulement en pilote) : **11 682 pages « oui »** en base
  (`validllm`/`validmodel` renseignés). Les verdicts « non » ont forcé la pertinence à 0, puis
  la suppression contrôlée les a retirés — d'où l'absence de « non » résiduels.
- `land delete --maxrel=1` : le land est passé de **207 366** expressions (instantané fin juin,
  §5.2) à **165 564** (audit du 04/07). L'écart (−41 802) cumule suppression maxrel, prune des
  liens malformés et fusions de doublons (§12).
- **282 pages à pertinence 0 subsistent** (recalculs `consolidate` postérieurs à la
  suppression) : sans effet sur les exports, tous filtrés `--minrel=1`.

In [ ]:
# Gate LLM — pilote (200 pages) pour vérifier verdicts et coût, puis passage complet
!python mywi.py land llm validate --name={LAND} --limit=200
!python mywi.py land llm validate --name={LAND}

In [ ]:
# Audit : verdicts LLM et distribution de pertinence
display(sql("""SELECT validllm, validmodel, COUNT(*) AS n FROM expression
               WHERE land_id = 1 AND validllm IS NOT NULL GROUP BY validllm, validmodel"""))
display(sql("""SELECT SUM(relevance >= 1) AS rel_sup_1, SUM(relevance = 0) AS rel_0
               FROM expression WHERE land_id = 1 AND fetched_at IS NOT NULL"""))

### Suppression contrôlée

> ⚠️ Triple garde-fou appliqué : (1) backup préalable, (2) aperçu SQL du périmètre exact,
> (3) confirmation stricte `Y` majuscule (pipée ici en connaissance de cause).

In [ ]:
# Suppression effective des pages crawlées à pertinence < 1 (confirmation 'Y' pipée)
!echo "Y" | python mywi.py land delete --name={LAND} --maxrel=1

> 💡 **Note de design** : le corpus apparié doit rester *constant* entre les deux procédures
> d'extraction — le nettoyage définit le **corpus qualifié** une fois pour toutes, *avant* la
> construction des deux réseaux. Toute suppression ultérieure invaliderait la comparaison (les
> deux réseaux ne partageraient plus le même ensemble de nœuds).

<a id="sec9"></a>
## §9 — Domaines et heuristiques

L'agrégation au niveau **entité web** repose sur la table `domain`. **Exécuté** : `domain crawl`
(titre, description, mots-clés de chaque page d'accueil ; chaîne Trafilatura → Wayback →
requests) — **25 010 domaines fetchés** sur 38 989 en table. La commande est **globale** (tous
lands confondus : un domaine est une entité partagée entre projets). Le corpus qualifié mobilise
**6 593 domaines distincts** (pertinence ≥ 1). `heuristic update` ré-applique les regex de
rattachement expression → domaine (idempotent ; pas de trace datée en base).

In [ ]:
!python mywi.py domain crawl --limit=500
# Relance sur échecs : --http=ERR (matche ERR_TRAFI, ERR_ARCHIVE*, ERR_ALL_FAILED, ARC_NO_HTML, REQ_NO_HTML, 000…)

In [ ]:
# heuristic update affiche le nombre de réattributions (« 0 domain(s) updated » si rien à corriger)
!python mywi.py heuristic update

In [ ]:
# Audit : couverture de la table domain + concentrations du corpus qualifié
display(sql("SELECT COUNT(*) AS domaines, SUM(fetched_at IS NOT NULL) AS fetches FROM domain"))
display(sql("""SELECT d.name, COUNT(*) AS pages
               FROM expression e JOIN domain d ON d.id = e.domain_id
               WHERE e.land_id = 1 AND e.relevance >= 1
               GROUP BY d.name ORDER BY pages DESC LIMIT 15"""))

> 💡 **Lecture critique** : repérer les concentrations suspectes. Si un seul domaine
> (agrégateur, plateforme) représente une part démesurée du corpus, c'est un biais de structure
> à documenter — voire un candidat à l'exclusion motivée. Le nombre de domaines distincts est
> l'estimation brute du **[M]** du papier (avant agrégation fine en entités web).

<a id="sec10"></a>
## §10 — Enrichissement SEO Rank

`land seorank` interroge l'API seo-rank.my-addr.com et stocke le JSON brut dans
`expression.seorank` : rang Moz, trafic estimé, backlinks, métriques sociales — attributs de
nœuds dans les exports (colonnes `sr_*`), utiles pour pondérer la visibilité des acteurs.
Filtres par défaut : `http_status=200`, pertinence ≥ 1, pages non déjà enrichies (reprise
incrémentale). Prérequis : `settings.seorank_api_key` ou `MWI_SEORANK_API_KEY`.

**Constat : 14 188 pages enrichies** sur 20 806 en HTTP 200 (≈ 68 %) — couverture partielle.

> ⚠️ À trancher : compléter (relancer la commande, ~1 s/page → compter ~2 h pour le reliquat)
> ou documenter le critère d'arrêt.

In [ ]:
# Pilote (graines, 100 pages), puis passage complet — 14 188 pages couvertes au 04/07
!python mywi.py land seorank --name={LAND} --depth=0 --limit=100
!python mywi.py land seorank --name={LAND}

<a id="sec11"></a>
## §11 — Médias : recensés, non analysés

Le crawl et le pipeline readable ont **recensé 19 366 médias** (19 363 images, 3 vidéos —
table `media`). **`land medianalyse` n'a pas été exécuté** : 0 média analysé (dimensions,
couleurs, EXIF, hash perceptuel absents).

> 📌 **Décision à acter** : hors périmètre d'Art. 1 (réseaux hypertexte — l'analyse visuelle
> n'entre dans aucun des trois gestes de la démonstration). À lancer seulement si un usage aval
> le justifie : `land medianalyse --name=airegulation --minrel=1` (télécharge chaque fichier —
> auditer le volume avant).

In [ ]:
# Statistiques agrégées du corpus média (lecture seule, sans téléchargement)
!python mywi.py land media_stats --name={LAND}

<a id="sec12"></a>
## §12 — Consolidation et dédup des URL (03–04/07/2026)

`land consolidate` reconstruit liens et médias depuis le contenu stocké et recalcule la
pertinence. Sur ce corpus, la séquence consolidation + dédup a été **l'épisode critique du
pipeline** : elle a réconcilié les deux définitions du réseau B.

**Chronologie** (preuves : fichiers `data/`, commits du clone, journal de developpement interne) :

| Date | Événement |
|---|---|
| 03/07 20:31 | Premier export fullhtml — **pré-consolidation, écarté de l'analyse** |
| 03/07 21:07 | **Backup** `mwi.db.bak_20260703_210650` (8,9 Go), puis prune des liens malformés (`scripts/prune_malformed_links.py`) + consolidate ; écritures jusqu'à ~01:00 |
| 04/07 07:27 + 08:18 | Lot apparié n°1 (fullhtml + body, 22 873 nœuds). **Diagnostic** : `pageslinks` = 8 043 (self-loops inclus), `in_mwi=1` = 5 598, `citation=1` = 8 902 → `in_mwi` sous-capturait **~37 %** des actes citationnels. Cause : `ExpressionLink` pointait des **doublons d'URL** (slash/scheme/www) à pertinence NULL, exclus par minrel, quand la passe citation résolvait en 3 clés vers le nœud canonique |
| 04/07 12:35–12:46 | Correctifs code : `fc2f6b3` (export/normalize sans self-loops, dédup des groupes de collision) et `22f7736` (**consolidate résout les variantes d'URL**, plus jamais de self-loop) ; consolidate ré-exécuté (base retouchée jusqu'à 14:02) |
| 04/07 13:35 + 13:45 | **Lot apparié n°2** (body + fullhtml, 21 430 nœuds) : écart provenance/citation tombé à **53 arêtes (0,65 %)** — les deux définitions de B convergent |
| 04/07 14:15 | Commit `cd850ea` (fusion des doublons exacts legacy = la cible des 53 résiduelles) — **postérieur** au lot n°2 → re-export final requis (§15) |

Effets de la dédup : A 24 773 → **22 778** arêtes ; B provenance 5 598 → **8 152** (+2 554 actes
citationnels récupérés) ; nœuds 22 873 → **21 430**.

> 📌 **Règles opératoires arrêtées** (à reporter au pré-enregistrement P0.10 et au DMP P0.8) :
> (1) **figer la base** avant tout export apparié — aucun crawl/readable/consolidate entre A et
> B ; (2) exporter A et B **dos à dos** et consigner horodatage + compteurs ; (3) **backup avant
> toute opération destructive** (normalize, delete, prune).

In [ ]:
# Consolidation : reconstruit expressionlink + médias depuis le contenu stocké, recalcule la pertinence.
# Dernière passe : 04/07/2026 (post-commits fc2f6b3/22f7736 — résolution des variantes d'URL)
!python mywi.py land consolidate --name={LAND} --depth=0

<a id="sec13"></a>
## §13 — Embeddings et pseudolinks : non exécutés

La chaîne sémantique (vectorisation des paragraphes → pseudolinks entre argumentaires proches)
**n'a pas été engagée** : tables `paragraph`, `paragraph_embedding`, `paragraph_similarity`
vides (0 partout). Couche pertinente pour l'aval (proximités discursives que les hyperliens ne
matérialisent pas), hors des trois gestes d'Art. 1. Le jour venu : `embedding check`, puis
`embedding generate --name=airegulation`, puis
`embedding similarity --name=airegulation --method=cosine --threshold=0.85 --minrel=1`.

<a id="sec14"></a>
## §14 — Tags : phase d'annotation à venir

Aucun tag posé (`tag` et `taggedcontent` vides) — conforme au protocole : l'annotation des
entités (catégories d'acteurs, positions discursives) est prévue **en phase d'analyse**, en
double codage aveugle dans l'interface MyWebClient, puis export ici
(`tag export --name=airegulation --type=matrix|content --minrel=1`).

<a id="sec15"></a>
## §15 — Exporter le corpus

**L'export pivot de l'étude** — le lot apparié, écrit dans `data/` sous le motif
`export_land_<land>_<type>_<horodatage>_*.csv` :

```bash
python mywi.py land export --name=airegulation --type=nodelinkcsv --minrel=1                  # B (body)
python mywi.py land export --name=airegulation --type=nodelinkcsv --fullhtml=TRUE --minrel=1  # A (fullhtml)
```

8 CSV (4 body + 4 fullhtml) sur le **même jeu de nœuds**, donc directement comparables. Colonnes
clés du `pageslinksfullhtml` : `weightbody`/`weighthtml` (partition **disjointe** par
construction — jamais les deux > 0) et `citation` (le lien figure dans le readable de la source,
résolution 3 clés exact/relaxed/host+path).

> ⚠️ **Pièges vérifiés sur ce corpus** : (1) la colonne `Weight` est **vide** — dans Gephi,
> utiliser `weightbody`/`weighthtml` ; (2) `wc -l` **surestime** les lignes (champs multilignes
> `context`/`description`) — toujours compter via un parseur CSV ; (3) filtre par défaut
> `--minrel=1` : les 282 pages à pertinence 0 sont silencieusement exclues.

### Journal des exports produits (état du disque au 04/07)

| Date | Fichier(s) | Statut |
|---|---|---|
| 27/06 09:31 | `nodelinkcsv` (4 fichiers body) | Pré-nettoyage, dépassé |
| 01/07 12:04 | `iaregulation.json` (graphe domaines) | Alimentation du viewer mwigraph |
| 03/07 10:46 | `pagesjson` | — |
| 03/07 20:31 | `nodelinkcsv` fullhtml | **Pré-consolidation, écarté** (seul `domainlinksfullhtml` subsiste sur disque) |
| 04/07 07:27 + 08:18 | Lot apparié n°1 | A servi au diagnostic de §12 ; fichiers retirés du disque |
| 04/07 13:35 + 13:45 | **Lot apparié n°2** (body + fullhtml) | **Référence actuelle** : 21 430 nœuds ; A = 22 778 ; B = 8 152 ; `citation=1` = 8 205 |
| 04/07 14:24 | `fullpagecsv` (267 Mo) | Post-dédup (texte readable complet) |

> 🔴 **Action ouverte — l'instantané final n'existe pas encore** : re-exporter le lot apparié
> **dos à dos, base figée, à HEAD ≥ `cd850ea`** (le commit qui fusionne les 53 doublons legacy
> est postérieur au lot n°2). Écart provenance/citation attendu ≈ 0. Consigner ici horodatage et
> compteurs : ce re-export sera l'instantané de l'article.

In [ ]:
# Le lot apparié — exporter les DEUX dos à dos, base figée (aucune écriture entre les deux)
!python mywi.py land export --name={LAND} --type=nodelinkcsv --minrel=1
!python mywi.py land export --name={LAND} --type=nodelinkcsv --fullhtml=TRUE --minrel=1

In [ ]:
# Export du texte intégral (fait le 04/07 14:24 — 267 Mo)
!python mywi.py land export --name={LAND} --type=fullpagecsv --minrel=1

In [ ]:
# Inventaire des fichiers produits
exports = sorted(DATA.glob(f"export_land_{LAND}_*"))
for f in exports:
    print(f"{f.stat().st_size/1024:12.1f} Ko  {f.name}")

In [ ]:
# Vérification des compteurs du dernier lot apparié — TOUJOURS via parseur CSV (jamais wc -l).
# Attendu sur le lot du 04/07 13:45 : A = 22 778 ; B = 8 152 ; citation = 8 205 ; self-loops = 0.
import csv

def latest(pattern):
    files = sorted(DATA.glob(pattern))
    return files[-1] if files else None

fh = latest(f"export_land_{LAND}_nodelinkcsv_*_pageslinksfullhtml.csv")
fb = latest(f"export_land_{LAND}_nodelinkcsv_*_pageslinks.csv")

if fh:
    A = B = CIT = SELF = 0
    with open(fh, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            A += 1
            if float(row["weightbody"] or 0) > 0: B += 1
            if row["citation"] == "1": CIT += 1
            if row["Source"] == row["Target"]: SELF += 1
    print(f"{fh.name}")
    print(f"  A (arêtes fullhtml) = {A:>6}")
    print(f"  B (weightbody > 0)  = {B:>6}")
    print(f"  citation = 1        = {CIT:>6}   (écart vs B : {CIT - B})")
    print(f"  self-loops          = {SELF:>6}   (attendu : 0)")
else:
    print("Aucun pageslinksfullhtml trouvé — lancer l'export apparié.")
if fb:
    with open(fb, newline="", encoding="utf-8") as f:
        nb = sum(1 for _ in csv.DictReader(f))
    print(f"{fb.name}")
    print(f"  pageslinks (body apparié) = {nb:>6}   (doit égaler B)")

> 💡 Le `pageslinksfullhtml` porte à lui seul le dispositif : **A** = toutes les lignes ;
> **B** = `weightbody > 0` ; **liens éliminés A∖B** = `weightbody = 0` (14 626 arêtes au lot
> n°2, dont **87,0 % intra-domaine** — signature gabarit/navigation — contre 32,8 % côté B).
> La prédiction directionnelle de H2 se lit déjà dans les comptes bruts : linkedin.com,
> euaiact.com, whitecase.com en tête des cibles éliminées ; artificialintelligenceact.eu,
> europarl.europa.eu, digital-strategy.ec.europa.eu en tête des cibles citationnelles.

<a id="sec16"></a>
## §16 — Bilan chiffré et reproductibilité

Les chiffres qui rempliront les crochets `[N]`, `[M]`, `[dates]` du manuscrit — re-dérivables à
tout moment par la cellule suivante.

In [ ]:
# Fiche d'identité chiffrée du corpus (expressionlink est global : base mono-land)
recap = sql("""
SELECT
  (SELECT COUNT(*) FROM expression e JOIN land l ON l.id=e.land_id
    WHERE l.name = :land)                                                       AS expressions_totales,
  (SELECT COUNT(*) FROM expression e JOIN land l ON l.id=e.land_id
    WHERE l.name = :land AND e.fetched_at IS NOT NULL)                          AS N_pages_crawlees,
  (SELECT COUNT(*) FROM expression e JOIN land l ON l.id=e.land_id
    WHERE l.name = :land AND e.relevance >= 1)                                  AS N_pages_qualifiees,
  (SELECT COUNT(DISTINCT e.domain_id) FROM expression e JOIN land l ON l.id=e.land_id
    WHERE l.name = :land AND e.relevance >= 1)                                  AS M_domaines,
  (SELECT MIN(DATE(e.fetched_at)) FROM expression e JOIN land l ON l.id=e.land_id
    WHERE l.name = :land)                                                       AS date_debut_crawl,
  (SELECT MAX(DATE(e.fetched_at)) FROM expression e JOIN land l ON l.id=e.land_id
    WHERE l.name = :land)                                                       AS date_fin_crawl,
  (SELECT COUNT(*) FROM expressionlink)                                         AS liens_editoriaux_bruts,
  (SELECT COUNT(*) FROM expression e JOIN land l ON l.id=e.land_id
    WHERE l.name = :land AND e.html IS NOT NULL)                                AS pages_html_archive,
  (SELECT COUNT(*) FROM expression e JOIN land l ON l.id=e.land_id
    WHERE l.name = :land AND e.validllm IS NOT NULL)                            AS gate_llm_oui,
  (SELECT COUNT(*) FROM expression e JOIN land l ON l.id=e.land_id
    WHERE l.name = :land AND e.seorank IS NOT NULL)                             AS seorank_enrichies
""", {"land": LAND})
recap.T.rename(columns={0: "valeur"})

### La checklist de reproductibilité (état réel)

| Exigence | Trace | État |
|---|---|---|
| Requêtes de collecte (texte, moteurs, fenêtres, dates) | **Ce notebook, §4** — les tables `searchquery`/`searchresultlog` sont vides (`land urlist` ne journalise pas en base) | ⚠️ Journal manuel — compléter Q3/Q4 |
| Provenance moteur de chaque graine | Non disponible (limite de la voie legacy) | ❌ À documenter comme limite |
| URL d'origine avant canonicalisation | `expression.original_url` (121 842) | ✅ |
| Stratégie réseau par page | `expression.fetch_method` + `http_status` | ✅ |
| HTML brut pour rejouer les deux extractions | `expression.html` (20 699 pages) | ✅ |
| Verdicts de qualification | `relevance` + `validllm`/`validmodel` (11 682) | ✅ |
| Backup avant opérations destructives | `data/mwi.db.bak_20260703_210650` (8,9 Go) | ✅ |
| Version de l'instrument épinglée | Clone `02_data/mwi`, HEAD `cd850ea` ; migrations 001-012 appliquées (**013 en attente**) | ⚠️ `db migrate` à relancer |
| Base figée + exports dos à dos | Règle §12 ; **re-export final à HEAD ≥ `cd850ea` en attente** | 🔴 Action ouverte |
| Dépôt des exports (DOI/Zenodo) | Non engagé (phase P5) | ❌ |

<a id="sec17"></a>
## §17 — Récapitulatif : le pipeline réellement exécuté, et le reste-à-faire

### Le pipeline exécuté (11/06 → 04/07/2026)

```bash
# §1-§3 — Instrument et projet (11/06)
python mywi.py db migrate
python mywi.py land create --name=airegulation --desc="Public debate and controversy…" --lang=en --fullhtml=TRUE
python mywi.py land addterm --land=airegulation --terms="artificial intelligence, AI act, …"   # 57 termes

# §4 — Graines : urlist ×2 moteurs, fenêtres mensuelles 06/2023 → 05/2026 (relances ×3)
python mywi.py land urlist --name=airegulation --query='…' --engine=google     --lang=en --datestart=2023-06-01 --dateend=2026-05-31 --timestep=month --sleep=1.0
python mywi.py land urlist --name=airegulation --query='…' --engine=duckduckgo --lang=en --datestart=2023-06-01 --dateend=2026-05-31 --timestep=month --sleep=1.0

# §5 — Crawl : rafales en terminal, PROFONDEUR 0 UNIQUEMENT (11-14/06, 27/06, 29-30/06)
for i in {1..100}; do python mywi.py land crawl --name="airegulation" --depth=0 --limit=100; done

# §6-§8 — Normalisation, readable, qualification, nettoyage
python mywi.py land normalize --name=airegulation --dry-run --verbose
python mywi.py land normalize --name=airegulation
python mywi.py db fix_archive_domains
python mywi.py land readable --name=airegulation --merge=smart_merge
python mywi.py land llm validate --name=airegulation
echo "Y" | python mywi.py land delete --name=airegulation --maxrel=1

# §9-§10 — Enrichissements
python mywi.py domain crawl --limit=500
python mywi.py heuristic update
python mywi.py land seorank --name=airegulation          # 14 188 pages (partiel)

# §12 — Maintenance 03-04/07 : backup, prune, consolidate, dédup URL (commits fc2f6b3, 22f7736, cd850ea)
python scripts/prune_malformed_links.py                  # après backup !
python mywi.py land consolidate --name=airegulation --depth=0

# §15 — Exports appariés dos à dos (base figée)
python mywi.py land export --name=airegulation --type=nodelinkcsv --minrel=1
python mywi.py land export --name=airegulation --type=nodelinkcsv --fullhtml=TRUE --minrel=1
python mywi.py land export --name=airegulation --type=fullpagecsv --minrel=1
```

### Reste-à-faire

| # | Action | Réf. |
|---|---|---|
| 1 | 🔴 **Re-export apparié final** à HEAD ≥ `cd850ea`, base figée, dos à dos ; consigner horodatage + compteurs ici | §15 |
| 2 | `db migrate` — appliquer la migration 013 (index composites) | §1 |
| 3 | Confirmer/consigner l'exécution de Q3-Q4 (et moteurs additionnels éventuels) | §4 |
| 4 | Décider du sort des 653 pages en 403 (relance `--retry-status` ou limite documentée) | §5 |
| 5 | SEO Rank : compléter les ~6 600 pages 200 restantes, ou documenter le critère d'arrêt | §10 |
| 6 | Acter « médias hors périmètre » (ou lancer `medianalyse`) | §11 |
| 7 | Entériner B := `weightbody`/`citation` convergents dans §4 Methods, après le re-export | §12, §15 |
| 8 | Dépôt Zenodo/DOI des exports + épingler le commit `cd850ea` dans le pré-enregistrement (P0.10) et le DMP (P0.8) | §16 |

### Les pièges rencontrés sur ce corpus (vécu, pas théorie)

| Piège | Parade appliquée |
|---|---|
| Doublons d'URL (slash/scheme/www/casse) → `in_mwi` sous-capturait 37 % des citations | Dédup en base (consolidate `22f7736`) + colonne `citation` ; convergence vérifiée à 0,65 % |
| `wc -l` sur les CSV (champs multilignes `context`/`description`) | Toujours compter via parseur CSV (§15) |
| Colonne `Weight` vide dans le fullhtml | Gephi : utiliser `weightbody`/`weighthtml` |
| Exports à base non figée → « instabilité » apparente des compteurs | Base figée + exports dos à dos + horodatage consigné |
| `--minrel=1` silencieux à l'export | Le corpus d'analyse = pertinence ≥ 1 (282 pages à 0 exclues) |
| `db setup` destructif ; confirmations `Y` strictes | `db migrate` seul ; backup avant delete/normalize/prune |
| `land urlist` ne journalise pas les requêtes en base | Ce notebook = journal ; requêtes consignées en §4 |

### Références

- Mémoire projet (audit-trail détaillé, journal interne non publié) — entrées du 04/07/2026 ;
- Architecture de l'article : `01_archi/00_ARCHITECTURE.md` (§4 corpus, §6 démonstration) ;
- Manuscrit : `04_redaction/Copie de Art1_Does_Extraction_Level_Matter_v4.md` (§4 Methods) ;
- Tutoriels MWI (référence générale) : `02_data/mwi_tutorial.md`, `mwi_tutorial_install.md`,
  `mwi_tutorial_crawl.md` — les docs du dépôt ont été supprimées au commit `c60246c`.